In [1]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.tensorflow

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    TextVectorization, Embedding, LSTM, Bidirectional,
    Dense, Dropout, RepeatVector, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Data
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = pd.DataFrame(dataset["train"])

X = df["text"].astype(str).values
y = df["label"].astype(int).values

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Text vectorization
max_words = 10000
seq_len = 50

vectorizer = TextVectorization(
    max_tokens=max_words,
    output_sequence_length=seq_len
)

vectorizer.adapt(X_train)

# MLflow experiment
mlflow.set_experiment("Tweet_Sentiment_LSTM_Models")

all_predictions = {}

def train_and_evaluate(model, name):

    # Start MLflow run for this model
    with mlflow.start_run(run_name=name):

        # Parameters
        epochs = 10
        batch_size = 64
        optimizer = "adam"
        loss_function = "sparse_categorical_crossentropy"

        # Log parameters to MLflow
        mlflow.log_param("model_name", name)
        mlflow.log_param("max_words", max_words)
        mlflow.log_param("seq_len", seq_len)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("optimizer", optimizer)
        mlflow.log_param("loss_function", loss_function)

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )

        model.compile(
            optimizer=optimizer,
            loss=loss_function,
            metrics=["accuracy"]
        )

        # Train the model
        history = model.fit(
            X_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=0.1,
            # callbacks=[early_stop],
            verbose=1
        )

        # Evaluate the model
        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

        y_pred = np.argmax(model.predict(X_test), axis=1)
        acc = accuracy_score(y_test, y_pred)

        all_predictions[name] = y_pred

        # Log metrics to MLflow
        mlflow.log_metric("test_loss", test_loss)
        mlflow.log_metric("test_accuracy", test_accuracy)
        mlflow.log_metric("accuracy_score", acc)

        # Log training history to MLflow
        for epoch, value in enumerate(history.history["loss"]):
            mlflow.log_metric("train_loss", value, step=epoch)

        for epoch, value in enumerate(history.history["accuracy"]):
            mlflow.log_metric("train_accuracy", value, step=epoch)

        for epoch, value in enumerate(history.history["val_loss"]):
            mlflow.log_metric("val_loss", value, step=epoch)

        for epoch, value in enumerate(history.history["val_accuracy"]):
            mlflow.log_metric("val_accuracy", value, step=epoch)

        # Log the model itself
        mlflow.tensorflow.log_model(
            model=model,
            name=f"{name}_model"
        )

        print(f"\nRun completed for {name}. Accuracy: {acc:.4f}")

        return acc


# LSTM
lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=max_words, output_dim=64),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Bi-LSTM
bilstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=max_words, output_dim=64),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Encoder-Decoder LSTM
ed_lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=max_words, output_dim=64),

    # Encoder
    LSTM(64),

    # Decoder
    RepeatVector(seq_len),
    LSTM(64, return_sequences=True),

    # Classification
    GlobalAveragePooling1D(),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Training
results = {}

results["LSTM"] = train_and_evaluate(lstm_model, "LSTM")
results["Bi-LSTM"] = train_and_evaluate(bilstm_model, "Bi-LSTM")
results["ED-LSTM"] = train_and_evaluate(ed_lstm_model, "ED-LSTM")

# Overall results
results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])

print("\n=== Overall Results ===")
print(results_df)

# Detailed results
label_names = ["negative", "neutral", "positive"]

for model_name, y_pred in all_predictions.items():
    print("\n" + "="*50)
    print(model_name)
    print("="*50)

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)

    summary = pd.DataFrame({
        "Class": label_names,
        "Correctly predicted": cm.diagonal(),
        "Total in test set": cm.sum(axis=1),
        "Accuracy per class": cm.diagonal() / cm.sum(axis=1)
    })

    print("\nDetailed class summary:")
    print(summary)

print("\nMLflow runs saved in the 'mlruns' folder.")

Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 35s 59ms/step - accuracy: 0.4481 - loss: 1.0195 - val_accuracy: 0.4507 - val_loss: 1.0173
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 31s 59ms/step - accuracy: 0.4512 - loss: 0.9857 - val_accuracy: 0.4460 - val_loss: 0.9619
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 43s 62ms/step - accuracy: 0.4635 - loss: 0.9590 - val_accuracy: 0.4732 - val_loss: 0.9753
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 35s 69ms/step - accuracy: 0.4585 - loss: 0.9795 - val_accuracy: 0.4792 - val_loss: 0.9956
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 38s 62ms/step - accuracy: 0.4820 - loss: 0.9951 - val_accuracy: 0.4805 - val_loss: 0.9963
Epoch 6/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 32s 61ms/step - accuracy: 0.4850 - loss: 0.9930 - val_accuracy: 0.4863 - val_loss: 0.9933
Epoch 7/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 43s 64ms/step - accuracy: 0.4864 - loss: 0.9936 - val_accuracy: 0.4858 - val_loss: 0.9909
Epoch 8/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 125s 243ms/step - accuracy: 0.4900 - loss: 0.9914 

2026/06/02 22:13:54 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Run completed for LSTM. Accuracy: 0.5811
Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - accuracy: 0.5903 - loss: 0.8576 - val_accuracy: 0.6342 - val_loss: 0.7846
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 25s 49ms/step - accuracy: 0.7092 - loss: 0.6616 - val_accuracy: 0.6279 - val_loss: 0.8028
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 25s 49ms/step - accuracy: 0.7555 - loss: 0.5716 - val_accuracy: 0.6205 - val_loss: 0.9086
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 37s 72ms/step - accuracy: 0.7935 - loss: 0.4923 - val_accuracy: 0.6079 - val_loss: 1.0932
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 26s 50ms/step - accuracy: 0.8210 - loss: 0.4294 - val_accuracy: 0.5959 - val_loss: 1.2107
Epoch 6/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 32s 62ms/step - accuracy: 0.8429 - loss: 0.3816 - val_accuracy: 0.5973 - val_loss: 1.3588
Epoch 7/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 25s 49ms/step - accuracy: 0.8603 - loss: 0.3395 - val_accuracy: 0.6118 - val_loss: 1.4238
Epoch 8/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 32s 61ms

2026/06/02 22:19:22 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Run completed for Bi-LSTM. Accuracy: 0.5941
Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 52s 92ms/step - accuracy: 0.4461 - loss: 1.0194 - val_accuracy: 0.4507 - val_loss: 1.0169
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 49s 95ms/step - accuracy: 0.4498 - loss: 1.0058 - val_accuracy: 0.4578 - val_loss: 0.9315
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.4822 - loss: 0.9365 - val_accuracy: 0.4912 - val_loss: 0.9778
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.4813 - loss: 0.9778 - val_accuracy: 0.5312 - val_loss: 0.9386
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 47s 92ms/step - accuracy: 0.5770 - loss: 0.8441 - val_accuracy: 0.5868 - val_loss: 0.8660
Epoch 6/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.6673 - loss: 0.7296 - val_accuracy: 0.6027 - val_loss: 0.8454
Epoch 7/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 47s 92ms/step - accuracy: 0.7272 - loss: 0.6380 - val_accuracy: 0.6192 - val_loss: 0.8667
Epoch 8/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 47s 9

2026/06/02 22:28:07 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Run completed for ED-LSTM. Accuracy: 0.6178

=== Overall Results ===
     Model  Accuracy
0     LSTM  0.581059
1  Bi-LSTM  0.594103
2  ED-LSTM  0.617779

LSTM

Classification report:
              precision    recall  f1-score   support

    negative       0.51      0.24      0.32      1419
     neutral       0.55      0.66      0.60      4134
    positive       0.63      0.62      0.63      3570

    accuracy                           0.58      9123
   macro avg       0.56      0.51      0.52      9123
weighted avg       0.58      0.58      0.57      9123


Detailed class summary:
      Class  Correctly predicted  Total in test set  Accuracy per class
0  negative                  339               1419            0.238901
1   neutral                 2735               4134            0.661587
2  positive                 2227               3570            0.623810

Bi-LSTM

Classification report:
              precision    recall  f1-score   support

    negative       0.47      0.39 